# *CONFIG FOR THE SPARK SESSION AND IMPORTING NECESSARY PACKAGES.*

In [7]:
import time
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from dotenv import load_dotenv

load_dotenv()
def spark_session()-> SparkSession:
      return ( SparkSession.builder \
            .appName("pyspark_project") \
            .master("local[*]") \
            .config("spark.driver.memory","1g") \
            .config("spark.sql.shuffle.partitions","8") \
            .config("spark.ui.port","4040") \
            .config("spark.sql.repl.eagerEval.enabled", True) \
            .config("spark.sql.adaptive.enabled","true") \
            .config("spark.sql.coalescePartitions.enabled","true") \
            .config("spark.sql.adaptive.skewJoin.enabled","true") \
            .getOrCreate()
       )
spark = spark_session()

### *READING ALL THE NECESSARY TABLES INTO DATAFRAME OBJECTS.* 


In [8]:
table_names:tuple[str] = (
                     "public.actor",
                     "public.address",
                     "public.category",
                     "public.city",
                     "public.country",
                     "public.customer",
                     "public.film",
                     "public.film_actor",
                     "public.film_category",
                     "public.inventory",
                     "public.payment",
                     "public.rental"
   )

table_PKs:tuple[str] = (
                    "actor_id",
                    "address_id",
                    "category_id",
                    "city_id",
                    "country_id",
                    "customer_id",
                    "film_id",
                    "inventory_id",
                    "payment_id",
                    "rental_id"
)

def read_tables(dbtable:str,partionColumn:str):
     return ( spark.read \
          .format("jdbc") \
          .option("url",os.getenv("DB_URL"))\
          .option("dbtable",dbtable) \
          .option("user",os.getenv("DB_USER")) \
          .option("password",os.getenv("DB_PASSWORD")) \
          .option("driver","org.postgresql.Driver")\
          .option("inferSchema","true") \
          .option("partitionColumn",partionColumn) \
          .option("lowerBound","1") \
          .option("upperBound","3000") \
          .option("numPartitions","10") \
          .load()
     )

actor = read_tables(table_names[0], table_PKs[0])
address = read_tables(table_names[1], table_PKs[1])
category = read_tables(table_names[2], table_PKs[2])
city = read_tables(table_names[3], table_PKs[3])
country = read_tables(table_names[4], table_PKs[4])
customer = read_tables(table_names[5], table_PKs[5])
film = read_tables(table_names[6], table_PKs[6])
film_actor = read_tables(table_names[7], table_PKs[6])
film_category = read_tables(table_names[8], table_PKs[6])
inventory = read_tables(table_names[9], table_PKs[7])
payment = read_tables(table_names[10], table_PKs[8])
rental = read_tables(table_names[11], table_PKs[9])

### *1.) Output the number of movies in each category, sorted in descending order.*

In [139]:
# category -> film_category -> film(left join)
result1 = (
          category.alias("c").join(film_category.alias("fc"),
                                   F.col("c.category_id")==F.col("fc.category_id"),
                                   "left"
                              )
                             .join(film.alias("f"),
                                   F.col("fc.film_id")==F.col("f.film_id"),
                                   "left"
                              )
                             .groupBy("c.name")
                             .agg(
                               F.count(F.col("f.film_id")).alias("movies_in_category")
                             )
                             .select("name","movies_in_category")
                             .orderBy(F.desc("movies_in_category"))
)
result1.show(10)
spark.stop()

+-----------+------------------+
|       name|movies_in_category|
+-----------+------------------+
|     Sports|                74|
|    Foreign|                73|
|     Family|                69|
|Documentary|                68|
|  Animation|                66|
|     Action|                64|
|        New|                63|
|      Drama|                62|
|     Sci-Fi|                61|
|      Games|                61|
+-----------+------------------+
only showing top 10 rows



### *2.) Output the 10 actors whose movies rented the most, sorted in descending order.*

In [ ]:
# actor->film_actor->inventory->rental(left join)
result2  = ( 
           actor.alias("a").join(
                            film_actor.alias("fa"),
                            F.col("a.actor_id")==F.col("fa.actor_id"),
                            "left"
                          )
                          .join(inventory.alias("i"),
                                F.col("fa.film_id")==F.col("i.film_id"),
                                "left"
                                
                          )
                          .join(rental.alias("r"),
                                F.col("i.inventory_id")==F.col("r.inventory_id"),
                                "left"
                          )
                          .groupBy("a.actor_id","a.first_name","a.last_name")
                          .agg(
                            F.count(F.col("r.rental_id")).alias("rental_count")
                          )
                          .withColumn("full_name",
                                      F.concat_ws(" ", F.col("a.first_name"),F.col("a.last_name"))
                                      )
                          .select("actor_id","full_name","rental_count")
                          .orderBy(F.desc("rental_count"))

)

result2.show(10)
spark.stop()

+--------+------------------+------------+
|actor_id|         full_name|rental_count|
+--------+------------------+------------+
|     107|    GINA DEGENERES|         753|
|     181|    MATTHEW CARREY|         678|
|     198|       MARY KEITEL|         674|
|     144|ANGELA WITHERSPOON|         654|
|     102|       WALTER TORN|         640|
|      60|       HENRY BERRY|         612|
|     150|       JAYNE NOLTE|         611|
|      37|        VAL BOLGER|         605|
|      23|     SANDRA KILMER|         604|
|      90|      SEAN GUINESS|         599|
+--------+------------------+------------+
only showing top 10 rows



### *3.) Output the category of movies on which the most money was spent.*

In [72]:
# category->film_category->film->inventory->rental->payment(inner join)
result3 = (
          category.alias("c").join(
                              film_category.alias("fc"),
                              F.col("c.category_id")==F.col("fc.category_id"),
                              "inner"
                             )
                             .join(
                              film.alias("f"),
                              F.col("fc.film_id")==F.col("f.film_id"),
                              "inner"
                             )
                             .join(
                              inventory.alias("i"),
                              F.col("f.film_id")==F.col("i.film_id"),
                              "inner"
                             )
                             .join(
                              rental.alias("r"),
                              F.col("i.inventory_id")==F.col("r.inventory_id"),
                              "inner"
                             )
                             .join(
                               payment.alias("p"),
                               F.col("r.rental_id")==F.col("p.rental_id"),
                               "inner"
                             )
                             .groupBy("c.name")
                             .agg(
                               F.sum("p.amount").alias("total_amount")
                             )
                             .select("name","total_amount")
                             .orderBy(F.desc("total_amount"))
                             .limit(1)
)

result3.show(10)
spark.stop()

+------+------------+
|  name|total_amount|
+------+------------+
|Sports|     5314.21|
+------+------------+



### *4.) Output the names of movies that are not in the inventory.* 

In [84]:
# film -> inventory(anti join)
result4 = (
           film.alias("f").join(
                           inventory.alias("i"),
                           F.col("f.film_id")==F.col("i.film_id"),
                           "anti"
                          )
                          .select(F.col("title").alias("name_of_movie"))
)

result4.show(10)
spark.stop()


+--------------------+
|       name_of_movie|
+--------------------+
|      ALICE FANTASIA|
|       ARK RIDGEMONT|
|      CHOCOLATE DUCK|
|COMMANDMENTS EXPRESS|
|    CRYSTAL BREAKING|
|DELIVERANCE MULHO...|
|       BUTCH PANTHER|
|     CROWDS TELEMARK|
|         APOLLO TEEN|
|ARSENIC INDEPENDENCE|
+--------------------+
only showing top 10 rows



### *5.) Output the top 3 actors who have appeared most in movies in the “Children” category. If several actors have the same number of movies, output all of them.*

In [ ]:
# actor->film_actor->film->film_category->category(left join)
result5 = (
          actor.alias("a").join(
                           film_actor.alias("fa"),
                           F.col("a.actor_id")==F.col("fa.actor_id"),
                           "left"
                          )
                          .join(
                            film.alias("f"),
                            F.col("fa.film_id")==F.col("f.film_id"),
                            "left"
                          )
                          .join(
                            film_category.alias("fc"),
                            F.col("f.film_id")==F.col("fc.film_id"),
                            "left"
                          )
                          .join(
                            category.alias("c"),
                            F.col("fc.category_id")==F.col("c.category_id"),
                            "left"
                          )
                          .filter(F.col("c.name")=="Children")
                          .groupBy("a.actor_id","a.first_name","a.last_name")
                          .agg(F.count("a.actor_id").alias("appearences"))
                          .withColumn("full_name",F.concat_ws(" ",F.col("a.first_name"),F.col("a.last_name")))
                          .select("actor_id","full_name","appearences")
                          .orderBy(F.desc("appearences"))
)

result5.show(3)
spark.stop()

+--------+--------------+-----------+
|actor_id|     full_name|appearences|
+--------+--------------+-----------+
|      17|  HELEN VOIGHT|          7|
|      66|    MARY TANDY|          5|
|     127| KEVIN GARLAND|          5|
|      80|    RALPH CRUZ|          5|
|     140|   WHOOPI HURT|          5|
|     142|    JADA RYDER|          4|
|     109|SYLVESTER DERN|          4|
|     101|   SUSAN DAVIS|          4|
|      93| ELLEN PRESLEY|          4|
|      92|KIRSTEN AKROYD|          4|
+--------+--------------+-----------+
only showing top 10 rows



### *6.) Output cities with the number of active and inactive customers (active - customer.active = 1). Sort by the number of inactive customers in descending order.* 

In [108]:
result6 = (
          city.alias("ci").join(
                          country.alias('co'),
                          F.col("ci.country_id")==F.col("co.country_id"),
                          "left"
                         )
                         .join(
                           address.alias("a"),
                           F.col("ci.city_id")==F.col("a.city_id"),
                           "left"
                         )
                         .join(
                           customer.alias("c"),
                           F.col("a.address_id")==F.col("c.address_id")
                         )
                         .groupBy("ci.city_id","ci.city")
                         .agg(
                           F.sum(F.when(F.col("c.active")==1,1).otherwise(0)).alias("active_customers"),
                           F.sum(F.when(F.col("c.active")==0,1).otherwise(0)).alias("Inactive_customers")
                         )
                         .select("city_id","city","active_customers","Inactive_customers")
                         .orderBy(F.desc("Inactive_customers"))
)

result6.show(10)
spark.stop()

+-------+----------------+----------------+------------------+
|city_id|            city|active_customers|Inactive_customers|
+-------+----------------+----------------+------------------+
|    578|        Xiangfan|               0|                 1|
|    281|          Ktahya|               0|                 1|
|    495| Southend-on-Sea|               0|                 1|
|    356|       Najafabad|               0|                 1|
|    111|Charlotte Amalie|               0|                 1|
|    577|         Wroclaw|               0|                 1|
|    259|          Kamyin|               0|                 1|
|    407|       Pingxiang|               0|                 1|
|    283|      Kumbakonam|               0|                 1|
|     57|         Bat Yam|               0|                 1|
+-------+----------------+----------------+------------------+
only showing top 10 rows



### *7.) Output the category of movies that have the highest number of total rental hours in the cities (customer.address_id in this city), and that start with the letter “a”. Do the same for cities with a “-” symbol.*

In [ ]:
w_rank = (
            Window .partitionBy("city") \
                   .orderBy(F.desc("total_rental_hours"))
       )

result7_01 = (
          category.alias("ca").join(
                             film_category.alias("fc"),
                             F.col("ca.category_id")==F.col("fc.category_id"),
                             "left"
                             )
                             .join(
                               film.alias("f"),
                               F.col("fc.film_id")==F.col("f.film_id"),
                               "left"
                             )
                             .join(
                               inventory.alias("i"),
                               F.col("f.film_id")==F.col("i.film_id"),
                               "left"
                             )
                             .join(
                               rental.alias("r"),
                               F.col("i.inventory_id")==F.col("r.inventory_id"),
                               "left"
                             )
                             .join(
                               customer.alias("c"),
                               F.col("r.customer_id")==F.col("c.customer_id"),
                               "left"
                             )
                             .join(
                               address.alias("a"),
                               F.col("c.address_id")==F.col("a.address_id"),
                               "left"
                             )
                             .join(
                               city.alias("ci"),
                               F.col("a.city_id")==F.col("ci.city_id"),
                               "left"
                             )
                             .filter((F.col("ca.name").startswith("a")) | (F.col("ca.name").startswith("A")))
                             .groupBy(F.col("ci.city").alias("city"),F.col("ca.name").alias("category_name"))
                             .agg(
                                   F.sum(F.col("rental_duration")).alias("total_rental_hours")
                             )
                             .withColumn(
                                     "total_rental_hours_ranking",
                                     F.dense_rank().over(w_rank)
                             )
                             .filter(F.col("total_rental_hours_ranking") == 1)
                             .select("city","category_name","total_rental_hours","total_rental_hours_ranking").distinct()
                             .orderBy(F.desc("total_rental_hours"))


     )
 
result7_01.show(10)
spark.stop()

+----------------+-------------+------------------+--------------------------+
|            city|category_name|total_rental_hours|total_rental_hours_ranking|
+----------------+-------------+------------------+--------------------------+
|      Cape Coral|    Animation|                41|                         1|
|         Cuautla|       Action|                37|                         1|
|           Laiwu|       Action|                37|                         1|
|        Mannheim|    Animation|                37|                         1|
|         Bijapur|    Animation|                36|                         1|
|Charlotte Amalie|    Animation|                36|                         1|
|              Po|       Action|                36|                         1|
|       Pontianak|       Action|                35|                         1|
|         Cianjur|    Animation|                33|                         1|
|         Fontana|       Action|                32| 

In [9]:
w_rank = (
            Window .partitionBy("city") \
                   .orderBy(F.desc("total_rental_hours"))
       )

result7_02 = (
          category.alias("ca").join(
                             film_category.alias("fc"),
                             F.col("ca.category_id")==F.col("fc.category_id"),
                             "left"
                             )
                             .join(
                               film.alias("f"),
                               F.col("fc.film_id")==F.col("f.film_id"),
                               "left"
                             )
                             .join(
                               inventory.alias("i"),
                               F.col("f.film_id")==F.col("i.film_id"),
                               "left"
                             )
                             .join(
                               rental.alias("r"),
                               F.col("i.inventory_id")==F.col("r.inventory_id"),
                               "left"
                             )
                             .join(
                               customer.alias("c"),
                               F.col("r.customer_id")==F.col("c.customer_id"),
                               "left"
                             )
                             .join(
                               address.alias("a"),
                               F.col("c.address_id")==F.col("a.address_id"),
                               "left"
                             )
                             .join(
                               city.alias("ci"),
                               F.col("a.city_id")==F.col("ci.city_id"),
                               "left"
                             )
                             .filter(F.col("ci.city").contains("-"))
                             .groupBy(F.col("ci.city").alias("city"),F.col("ca.name").alias("category_name"))
                             .agg(
                                   F.sum(F.col("rental_duration")).alias("total_rental_hours")
                             )
                             .withColumn(
                                     "total_rental_hours_ranking",
                                     F.dense_rank().over(w_rank)
                             )
                             .filter(F.col("total_rental_hours_ranking") == 1)
                             .select("city","category_name","total_rental_hours","total_rental_hours_ranking").distinct()
                             .orderBy(F.desc("total_rental_hours"))


  )
 
result7_02.show(10)
spark.stop()

+--------------------+-------------+------------------+--------------------------+
|                city|category_name|total_rental_hours|total_rental_hours_ranking|
+--------------------+-------------+------------------+--------------------------+
|          Mwene-Ditu|       Travel|                36|                         1|
|         Saint-Denis|       Sci-Fi|                35|                         1|
| Kamjanets-Podilskyi|     Children|                30|                         1|
|      Kirovo-Tepetsk|       Family|                29|                         1|
|     Southend-on-Sea|       Sports|                29|                         1|
|Donostia-San Seba...|       Sports|                28|                         1|
|        Shahr-e Kord|      Foreign|                28|                         1|
|         Beni-Mellal|      Foreign|                25|                         1|
|    Shubra al-Khayma|       Sci-Fi|                24|                         1|
|   